# Minimal Example Unsupervised

This jupyter notebook serves to provide a minimal example of training an unsupervised neural-network quantum states

## Import library

In [3]:
import sys
sys.path.append(r'../')

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

## Set the same seed
np.random.seed(42)
tf.random.set_seed(42)

%matplotlib inline

2023-04-11 11:00:53.295366: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-04-11 11:00:53.304337: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/gpfs/loomis/apps/avx/software/miniconda/4.12.0/lib:/gpfs/loomis/apps/avx/software/miniconda/4.12.0/lib
2023-04-11 11:00:53.304358: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2023-04-11 11:00:55.153190: W external/org_tensorflow/tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcu

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Training

Here, we use the library for the application to find the ground state of a one-dimensional Ising model with $4$ particles with open boundary conditions. We set $J = 2$ and $h = 1$ as the parameters of the Ising model, which means that the model is in ferromagnetic phase.  We use Gibbs sampling with $1000$ samples. We use restricted Boltzmann machine for positive real wave function where $\alpha = 2$. The weights and biases are initialised from a random normal with zero-mean and $0.01$ standard deviation. We use Adam~\cite{kingma2014adam} optimiser with $0.001$ learning rate. For the stopping criterion, we set $\varepsilon_\sigma$ as $0.01$ and train for maximally $1000$ epochs. At the end of the training, we compute the $M^2_F$ observable. 

In [5]:
from graph import Hypercube
from hamiltonian import HeisenbergJ1J2
from sampler import MetropolisExchange
from functools import partial
from model import MLPComplexTanh
from learner import Learner
from logger import Logger

## Define the model
square2d = Hypercube(length=4, dimension=2, pbc=True, next_nearest=True)
hamil = HeisenbergJ1J2(square2d, j1=1.0, delta=1.0, j2=0.0, total_sz=0.0)
hamil.diagonalize()

## Define the sampler
sampler = MetropolisExchange(num_samples=5000)

## Define the neural networks model
initializer = partial(np.random.normal, loc=0.0, scale=0.01)
mlp = MLPComplexTanh(num_visible = square2d.num_points, density=2, initializer=initializer)

## Define hyperparameters for learner
learning_rate = 0.001
optimizer = tf.keras.optimizers.Adam(learning_rate)
stopping_threshold = 0.01
num_epochs = 1000

## Training Process
learner = Learner(hamiltonian = hamil, model = mlp, sampler = sampler, optimizer = optimizer,
                  num_epochs = num_epochs, stopping_threshold = stopping_threshold,
                  observables = [], reference_energy=hamil.get_gs_energy())
learner.learn()

## Logging process
# result_path = 'result/'
# subpath = 'heisenbergj1j2'
# logger = Logger(result_path, subpath)
# logger.log(learner)

: 

: 